---
title: "From Classical Models to General-Purpose Classifiers"
categories: [machine-learning, text-classification]
---


Text classification did not begin with transformers. This chapter maps the main representation and architecture choices from bag-of-words to general-purpose decision APIs, then returns to the repository's AI-detector comparisons. The historical examples and Jev discussion follow Sebastian Raschka's [article on classifiers from bag-of-words to Jev](https://magazine.sebastianraschka.com/p/classifier-history-and-jev). The sequence explains what each family makes easy, what information it can use, and what claim a comparison can support.

## Bag-of-words and classical classifiers

Bag-of-words converts a variable-length document into a fixed-size vector over a vocabulary. A count or TF–IDF value records how strongly a token or n-gram appears, while a classifier such as Naive Bayes, logistic regression, an SVM, or boosted trees maps those features to labels. The representation is sparse, quick to fit, and a strong baseline when words or short phrases predict the target. A plain bag loses word order: “the dog bites the man” and “the man bites the dog” have identical unigram counts. N-grams restore short local order while expanding the feature space.

This is why the course keeps the logistic-regression baseline even after introducing transformers. A fast baseline tests whether the task is learnable from surface features, gives a point of comparison for training and inference cost, and exposes shortcuts through feature inspection. It may be the best choice for a narrow, repeated task with stable labels and enough representative data.

## Neural text classifiers before transformers

A word embedding maps each token to a dense learned vector. Unlike bag-of-words, a sequence model can preserve order. Recurrent neural networks process tokens one at a time while updating a hidden state; LSTMs and GRUs use gates to regulate what information persists. The sequential dependency can make training and inference less parallel, and a fixed-size hidden state can bottleneck long contexts.

A text CNN applies filters to neighboring embedding windows, then pools the resulting features into a fixed-size document representation. Filters detect local patterns and can be computed in parallel. CNN and RNN classifiers remain useful reference architectures: they show that “neural” does not automatically mean “transformer,” and local or sequential inductive biases can fit some tasks efficiently.

Transfer learning changed the data requirement. ULMFiT demonstrated pretraining a language model and fine-tuning it for a target classification task before transformer encoders became dominant. Chapter 06 studies this pretrain-then-adapt pattern in the detector project.

## Transformer routes to a label

The original transformer is encoder-decoder. Classification can use an encoder such as BERT or ModernBERT with a task-specific head; prompt a decoder such as GPT to emit a label; fine-tune a decoder to emit a label or attach a classification head; or use an encoder-decoder such as T5 for text-to-text prediction or a classification head.

These methods expose different interfaces. An encoder head directly produces class logits from a pooled sequence representation. A prompted decoder produces tokens, so the prompt, decoding settings, and output parser become part of the system. A decoder classification head must use a representation that can causally attend to the full input, commonly the last non-padding token. T5 can formulate labels as generated text, but that text still needs to map reliably to the class schema.

The article's IMDb history reports illustrative, roughly balanced-dataset results from separate experiments: a bag-of-words logistic-regression model reached 89.9% accuracy, an LSTM 85.66%, a CNN 90.07%, ULMFiT 95.4%, ModernBERT about 95%, and GPT-2 about 92%. They are not a controlled leaderboard: training setups, tuning effort, hardware, and possibly split details differ. Read them as evidence that a simple baseline can be competitive and that pretraining can help, not as a ranking to copy into a model card.

## Task-specific, prompted, and general-purpose decisions

A task-specific classifier fixes its label set during training. A general LLM can be prompted for many one-off decisions, but generated text can add latency, cost, and format errors. A general-purpose decision model aims to accept new task instructions and candidate labels or criteria through one structured interface, so users can try new decisions without collecting a separate training set and fine-tuning a model for every task.

The article uses Jev as a contemporary case study. Its documented API surface is useful to reason about even when we cannot inspect its weights or training data:

| Interface | Output contract | Typical classification shape |
|---|---|---|
| Choice | Select one option from a supplied set, with a class distribution | Exclusive binary or multi-class classification |
| Noul | Answer one or more yes/no questions independently | Binary questions or multi-label classification; probabilities need not sum to one |
| Score | Assign a value on an ordered rubric | Ordinal classification |

A general-purpose interface could be valuable when tasks vary. Its broad usefulness is an empirical claim about transfer across tasks, not a consequence of the API shape. For a narrow high-volume problem, a fine-tuned classifier may remain cheaper, faster, more accurate, or easier to validate.

## A Jev-like scoring design

The article also describes one way to retrofit a flexible choice API onto a transformer. This is a design pattern for study, not a claim about Jev's undisclosed internals. Pair the input text with the task instruction and one candidate label description at a time. Let $\textbf{\textsf{h}}_i\in\mathbb{R}^d$ be the model representation for candidate $i$, $\textbf{\textsf{w}}\in\mathbb{R}^d$ a shared learned vector, and $b\in\mathbb{R}$ a scalar bias. A one-output head scores each candidate:

$$s_i=\textbf{\textsf{w}}^{\top}\textbf{\textsf{h}}_i+b.$$

Normalize the scores over the $K$ candidates with softmax:

$$p_i=\frac{\exp(s_i)}{\sum_{j=1}^{K}\exp(s_j)}.$$

The shared head has the same parameters when $K$ changes; the provided candidate descriptions define the current choice set. The model can thus score new label options without changing the output layer, although good performance on new tasks still depends on training and evaluation data. For an encoder, $\textbf{\textsf{h}}_i$ can come from a pooled or `[CLS]` representation. For a causal decoder, use a position that can attend to the complete input, such as its final non-padding token.

## What is known about Jev

The article says the model is proprietary and its exact architecture, training data, and training algorithm are not public. The author speculates that it could use a small encoder-style model and a calibrated-decision training method. That is a hypothesis, not a verified description. The article discusses Reinforcement Learning with Calibration Rewards (RLCR) as a related published research idea, but does not establish that Jev's named RLCD method is RLCR or works the same way.

The article's reported IMDb comparison is a useful protocol example: it includes prediction quality, API cost, runtime, and repeated-run variation, and explicitly notes possible test-set overlap with training data. Treat those numbers as a dated author-reported experiment, not an independent benchmark or proof of broad generalization. Chapter 08 develops the stronger evaluation checklist.

## Exercise: compare the model families

Choose three approaches: the repository's TF–IDF logistic regression, one fine-tuned transformer, and either a prompted LLM or a general-purpose decision API. Define one held-out text-classification task and use the same examples and label definitions where possible. For each system, record its input representation, output contract, training or prompt setup, cost, latency, and test metrics. If access to a decision API is unavailable, analyze the article's reported comparison and mark every unverified condition.

Then propose a second, meaningfully different task. Explain what evidence would justify calling a system “general-purpose” across those tasks. Include held-out task design, possible benchmark contamination, repeated runs, and a task-specific baseline in your protocol.